# 第41章　分布外検出と安全な棄却 ― 「診るべきでない画像」を見分ける

**『医療診断支援AI開発　第2巻　実装編 ― 本格実装から臨床導入まで（第2巻　実装編）』のコード**

本文に載っているコードを、章の順にそのまま収めています。紙面のコードは読んで理解するためのもの、こちらは動かすためのものです。

- Python 以外（シェル・YAML・Dockerfile など）は、実行環境が違うので**コードセルにせず、そのまま読める形で置いています**。使う場所を確かめてから実行してください。
- 抜粋である以上、上から順に実行するだけで通るとは限りません。データの取得先やパスは、お手元の環境に合わせてください。
- **教育・研究のためのコードです。患者データをこのノートブックに置かないでください。**

リポジトリ: https://github.com/kewel-corp/book-impl

## OODスコアを、実装で捉える

In [ ]:
import torch

# マハラノビス距離によるOODスコア（学習後に一度だけ統計を推定）
feats, labels = extract_penultimate(model, train_loader)   # 学習データの特徴とラベル
# 平均は「クラスごと」に取る。全データで一つだけ推定すると、クラスが分かれている
# 特徴空間では“クラスとクラスの谷間”が中心になり、そこに落ちたOODを見逃す。
missing = [c for c in range(num_classes) if not (labels == c).any()]
assert not missing, f"学習データに無いクラスがある: {missing}（平均が定義できない）"
mus = torch.stack([feats[labels == c].mean(0) for c in range(num_classes)])
centered = torch.cat([feats[labels == c] - mus[c] for c in range(num_classes)])
cov = torch.cov(centered.T)                          # 共分散はクラス共通（tied）で一つ
D = cov.shape[0]
# 特徴の次元に対して症例が少ないと、共分散は特異（逆行列が無い）になる。疑似逆行列（pinv）で済ませると、
# 学習データが動かなかった方向の重みが0になり、その方向に大きく外れた入力を、かえって「近い」と測ってしまう。
# 対角に小さな値を足して（縮小推定）から逆行列をとり、ランクと条件数は記録しておく
print("rank", int(torch.linalg.matrix_rank(cov)), "/", D, " cond", float(torch.linalg.cond(cov)))
lam = 1e-3 * torch.trace(cov) / D                    # 強さは検証データで確かめて決める
cov_inv = torch.linalg.inv(cov + lam * torch.eye(D))

def ood_score(x):
    z, _ = extract_penultimate(model, x)           # 入力の特徴（ラベル側は使わない）
    d = z[:, None, :] - mus[None, :, :]            # (B, C, D)
    m = torch.einsum('bcd,de,bce->bc', d, cov_inv, d)
    return m.min(dim=1).values                     # 最も近いクラスまでの距離＝OODスコア

## 誤り率を「保証」する ― コンフォーマル予測

In [ ]:
import numpy as np

# 不確実性と棄却の章の conformal_qhat と同じ関数（章単位で動かせるよう再掲。仕様を一つにそろえる）
def conformal_qhat(prob_cal, y_cal, alpha=0.1):
    prob_cal, y_cal = np.asarray(prob_cal, dtype=float), np.asarray(y_cal, dtype=int)
    n = len(y_cal)
    assert n > 0 and 0 < alpha < 1 and np.isfinite(prob_cal).all(), "入力が不正"
    s = np.sort(1 - prob_cal[np.arange(n), y_cal])        # 較正データの非適合スコア（昇順）
    k = int(np.ceil((n + 1) * (1 - alpha)))               # k 番目の順序統計量を閾値にする
    if k > n:                                             # 較正例が少なすぎる：閾値∞＝全クラスを返す
        return np.inf
    return float(s[k - 1])

qhat = conformal_qhat(softmax(model(cal_x)), cal_y, alpha=0.1)   # 較正データ（学習にもテストにも使わない）
def predict_set(x):
    p = softmax(model(x))
    return np.where(1 - p <= qhat, 1, 0)                  # 集合に含めるクラス（qhat=∞ なら全クラス）

## 不確実性を2つに分ける ― aleatoric と epistemic

In [ ]:
import torch

# 不確実性と棄却の章（12.3節）と同じ3関数。章単位で動かせるよう再掲
def enable_dropout(model):
    for m in model.modules():
        if m.__class__.__name__.startswith("Dropout"):
            m.train()                             # 推論時もドロップアウトを効かせる

def snapshot_modes(model):                        # module ごとの training フラグを控える
    return [(m, m.training) for m in model.modules()]

def restore_modes(snapshot):
    for m, flag in snapshot:
        m.train(flag)

def entropy(p, dim=-1, eps=1e-12):                # 確率分布 p のエントロピー（dim 方向に和）
    return -(p * (p + eps).log()).sum(dim)

# model.train() を呼んではいけない。BatchNormまで学習モードになり、
# 推論のたびに running統計が書き換わって、以後の通常推論の出力まで変質する。
T = 20                                            # サンプリング回数（x は (B, ...) の入力）
saved = snapshot_modes(model)
try:
    model.eval()
    enable_dropout(model)                         # Dropout モジュールだけ train に戻す
    with torch.no_grad():
        probs = torch.stack([model(x).softmax(-1) for _ in range(T)])  # (T, B, C)
finally:
    restore_modes(saved)                          # 例外が出ても、呼ぶ前の状態へ module 単位で戻す
mean_p = probs.mean(0)                            # (B, C)
total = entropy(mean_p)                           # 総不確実性 (B,)
aleatoric = entropy(probs).mean(0)                # 各予測のエントロピーの平均 (B,)
epistemic = total - aleatoric                     # 差分＝モデルの無知 (B,)